# 🤖 LLM Paper Q&A — Qwen + RAG

**实验组**：检索论文作为 context 后问 Qwen。

架构：`Question → Embed → FAISS Top-3 → Context → Qwen → Answer`

**运行顺序**：Cell 1 → 2 → 3 → 4 → 5（挂载/上传 papers.json） → 6 → 7 → 8 → 9（验证） → 10（演示） → 11 → 12 → 13

In [ ]:
!pip install -q transformers torch accelerate sentence-transformers faiss-cpu
import torch, time
from transformers import AutoModelForCausalLM, AutoTokenizer

In [ ]:
MODEL_NAME = 'Qwen/Qwen2.5-1.5B-Instruct'
MAX_NEW_TOKENS = 300
DO_SAMPLE = False
print('Loading ' + MODEL_NAME + ' ...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if torch.cuda.is_available():
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float16, device_map='auto')
    print('OK loaded (GPU).  VRAM: ' + str(round(torch.cuda.memory_allocated()/1024**2)) + ' MB')
else:
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float32, device_map=None)
    print('OK loaded (CPU).  Note: inference will be slow on CPU.')

In [ ]:
def normalize_text(text: str) -> str:
    import re
    text = text.lower()
    text = re.sub(r'[\-–—:.,;!?()\[\]{}]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def evaluate(result: dict) -> dict:
    a = result['answer']
    case = result['case']
    norm_a = normalize_text(a)

    kw_hit = sum(
        1 for kw in case['expected_keywords']
        if normalize_text(kw) in norm_a
    )
    kw_recall = kw_hit / max(len(case['expected_keywords']), 1)

    title_hit = 0.0
    if case['expected_paper_title']:
        title_hit = 1.0 if normalize_text(case['expected_paper_title']) in norm_a else 0.0

    import re
    has_year = bool(re.search(r'\b(19|20)\d{2}\b', a))
    has_author = bool(re.search(
        r'\b[A-Z][a-z]+(?:\s+[A-Z][a-z]+)*\s+et\s+al\.?|'
        r'\b[A-Z][a-z]+,\s*[A-Z]\.?\s*[A-Z]?\.?\b', a
    ))
    citation_marker_present = has_year or has_author

    unsure_count = sum(
        1 for w in [
            "i don't know", "i cannot", "i'm not sure",
            'as an ai', 'i am an ai', 'undefined', 'unknown model',
        ]
        if w in a.lower()
    )

    # FIX 4: retrieval_hit — 评估检索命中，不评估生成
    # 1 if expected title is exactly in retrieved_titles list, else 0
    retrieval_hit = 1.0 if (
        result.get('retrieved_titles') and
        case['expected_paper_title'] in result['retrieved_titles']
    ) else 0.0

    return {
        'keyword_recall': kw_recall,
        'keyword_hit': kw_hit,
        'keyword_total': len(case['expected_keywords']),
        'title_hit': title_hit,
        'citation_marker_present': citation_marker_present,
        'has_year': has_year,
        'has_author': has_author,
        'unsure_count': unsure_count,
        'chars': len(a),
        'words': len(a.split()),
        'latency_seconds': result.get('latency', 0.0),
        'retrieval_hit': retrieval_hit,
    }

In [ ]:
TEST_CASES = [
    {
        'id': 1,
        'question': 'Who introduced the Transformer architecture and in which year was the paper published?',
        'expected_paper_title': 'Attention Is All You Need',
        'expected_keywords': ['Vaswani', '2017', 'Transformer', 'attention', 'encoder', 'decoder'],
    },
    {
        'id': 2,
        'question': 'What is the main contribution of the LoRA paper?',
        'expected_paper_title': 'LoRA: Low-Rank Adaptation of Large Language Models',
        'expected_keywords': ['LoRA', 'low-rank', '2021', 'efficient', 'fine-tuning', 'rank-decomposition'],
    },
    {
        'id': 3,
        'question': 'What is Constitutional AI and which organization proposed it?',
        'expected_paper_title': 'Constitutional AI: Harmlessness from AI Feedback',
        'expected_keywords': ['Constitutional AI', 'Anthropic', '2022', 'self-critique', 'RLAIF'],
    },
    {
        'id': 4,
        'question': 'Which paper applied RLHF to instruction following?',
        'expected_paper_title': 'Training language models to follow instructions with human feedback',
        'expected_keywords': ['InstructGPT', 'RLHF', 'OpenAI', '2022', 'reward model', 'PPO'],
    },
    {
        'id': 5,
        'question': 'What is the LLaMA model and which company released it?',
        'expected_paper_title': 'LLaMA: Open and Efficient Foundation Language Models',
        'expected_keywords': ['LLaMA', 'Meta', '2023', 'foundation model', '65B', 'transformer'],
    },
    {
        'id': 6,
        'question': 'What is the main idea of the Atlas paper?',
        'expected_paper_title': 'Atlas: Few-shot Learning with Retrieval Augmented Language Models',
        'expected_keywords': ['Atlas', 'retrieval-augmented', '2022', 'Meta', 'few-shot', 'knowledge'],
    },
    {
        'id': 7,
        'question': 'What is the T5 model and what is its main contribution?',
        'expected_paper_title': 'Exploring the Limits of Transfer Learning with a Unified Text-to-Text Transformer',
        'expected_keywords': ['T5', 'Text-to-Text', '2019', 'Google', 'transfer learning', 'Unified'],
    },
    {
        'id': 8,
        'question': 'Explain chain-of-thought prompting and which paper introduced it.',
        'expected_paper_title': 'Chain-of-Thought Prompting Elicits Reasoning in Large Language Models',
        'expected_keywords': ['chain-of-thought', 'CoT', '2022', 'reasoning', 'emergent ability', 'arithmetic'],
    },
    {
        'id': 9,
        'question': 'What is the difference between LoRA and QLoRA?',
        'expected_paper_title': 'QLoRA: Efficient Finetuning of Quantized LLMs',
        'expected_keywords': ['QLoRA', 'LoRA', 'quantization', '4-bit', '2023', 'Dettmers'],
    },
    {
        'id': 10,
        'question': 'What does the Self-RAG paper propose and how does it differ from standard RAG?',
        'expected_paper_title': 'Self-RAG: Learning to Retrieve, Generate, and Critique through Self-Reflection',
        'expected_keywords': ['Self-RAG', 'self-reflection', '2024', 'RAG', 'retrieve', 'reflection tokens'],
    },
]
print(str(len(TEST_CASES)) + ' test cases loaded')

In [ ]:
import json, os

LOCAL_PATH = '/content/papers.json'
DRIVE_PATH = '/content/drive/MyDrive/paper_rag/papers.json'

if os.path.exists(LOCAL_PATH):
    PAPERS_PATH = LOCAL_PATH
    print('Using local: ' + PAPERS_PATH)
    with open(PAPERS_PATH, 'r', encoding='utf-8') as f:
        PAPERS = json.load(f)
    print('OK loaded ' + str(len(PAPERS)) + ' papers')
else:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except ImportError:
        raise FileNotFoundError(
            'papers.json not found at ' + LOCAL_PATH + '. '
            'Please upload papers.json to the working directory '
            '(or run in Google Colab to use Drive).'
        )
    if os.path.exists(DRIVE_PATH):
        PAPERS_PATH = DRIVE_PATH
        print('Using Drive: ' + PAPERS_PATH)
        with open(PAPERS_PATH, 'r', encoding='utf-8') as f:
            PAPERS = json.load(f)
        print('OK loaded ' + str(len(PAPERS)) + ' papers')
    else:
        raise FileNotFoundError(
            'papers.json not found at ' + LOCAL_PATH + ' or ' + DRIVE_PATH + '. '
            'Please upload papers.json to Colab or mount Google Drive.'
        )

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np, faiss
EMBED_DEVICE = 'cpu'
print('Embedding device: ' + EMBED_DEVICE)
embed_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2', device=EMBED_DEVICE)
texts = [p['title'] + '. ' + p['abstract'] for p in PAPERS]
embs = embed_model.encode(texts, normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=False)
dim = embs.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(embs.astype('float32'))
print('OK FAISS: ' + str(index.ntotal) + ' vectors, dim=' + str(dim))

In [ ]:
def retrieve(question: str, k: int = 3) -> list:
    if not PAPERS or index.ntotal == 0:
        raise RuntimeError('retrieve() called with empty corpus: PAPERS=' + str(len(PAPERS)) + ', index.ntotal=' + str(index.ntotal))
    try:
        k_int = int(k)
        if k_int <= 0:
            raise ValueError('k must be positive, got ' + str(k))
    except (TypeError, ValueError) as e:
        raise ValueError('retrieve() k must be a positive integer, got ' + repr(k)) from e
    max_k = min(len(PAPERS), index.ntotal)
    k_int = max(1, min(k_int, max_k))
    q_emb = embed_model.encode([question], normalize_embeddings=True, convert_to_numpy=True)
    scores, ids = index.search(q_emb.astype('float32'), k_int)
    hits = []
    for s, i in zip(scores[0], ids[0]):
        if i < 0 or i >= len(PAPERS):
            continue
        hits.append({
            'score': float(s),
            'title': PAPERS[i]['title'],
            'authors': PAPERS[i]['authors'],
            'year': PAPERS[i]['year'],
            'abstract': PAPERS[i]['abstract'],
        })
    return hits

In [ ]:
def get_context(question: str) -> dict:
    K = min(3, len(PAPERS))
    hits = retrieve(question, k=K)
    if not hits:
        return {'text': 'No relevant context found.', 'retrieved_titles': []}
    titles = [h['title'] for h in hits]
    text_parts = []
    for h in hits:
        authors = h['authors']
        if isinstance(authors, list):
            authors = ', '.join(authors)
        abstract_snippet = (h['abstract'] or '')[:600]
        text_parts.append(
            '[' + h['title'] + ' (' + str(h['year']) + ') by ' + authors + ']\n' + abstract_snippet
        )
    return {'text': '\n\n'.join(text_parts), 'retrieved_titles': titles}

In [ ]:
# RAG 验证 + preflight
# 第一步：globals 缺失检查（按 cell 顺序）
missing_vars = []
rag_var_map = {
    'PAPERS':     'Cell 5 (drive + papers)',
    'embed_model':'Cell 6 (embedding)',
    'index':      'Cell 6 (FAISS)',
    'retrieve':   'Cell 7 (retrieve)',
    'get_context':'Cell 8 (get_context)',
}
for _var, _label in rag_var_map.items():
    if _var not in dir():
        missing_vars.append(_var + ' - run ' + _label + ' first.')
if missing_vars:
    raise RuntimeError('RAG preflight failed (globals):\n  ' + '\n  '.join(missing_vars))

# 第二步：字段完整性
assert len(PAPERS) >= 10, 'Need >=10 papers, got ' + str(len(PAPERS))
for i, p in enumerate(PAPERS):
    for field in ['title', 'authors', 'year', 'abstract']:
        assert field in p and p[field], 'Paper ' + str(i) + ' missing/non-empty: ' + field

# 第三步：ntotal 匹配
assert index.ntotal == len(PAPERS), 'FAISS ntotal=' + str(index.ntotal) + ' != len(PAPERS)=' + str(len(PAPERS))

# 第四步：expected title 命中率（必须 10/10）
expected_titles = [tc['expected_paper_title'] for tc in TEST_CASES]
matched = sum(1 for t in expected_titles if any(p['title'] == t for p in PAPERS))
missing_titles = [t for t in expected_titles if not any(p['title'] == t for p in PAPERS)]
assert matched == len(expected_titles), (
    'preflight title check: ' + str(matched) + '/' + str(len(expected_titles)) + ' expected titles found. '
    'Missing: ' + str(missing_titles)
)
print('RAG validation: ' + str(len(PAPERS)) + ' papers, index=' + str(index.ntotal) + ', '
      + str(matched) + '/' + str(len(expected_titles)) + ' expected titles found')
print('RAG preflight: OK')

In [ ]:
# RAG 演示（在初始化完成后运行）
print('=== RAG retrieval demo ===')
demo_q = 'Who introduced the Transformer architecture?'
hits = retrieve(demo_q, k=3)
for h in hits:
    print('  [' + str(round(h['score'], 3)) + '] ' + h['title'] + ' (' + str(h['year']) + ')')
ctx = get_context(demo_q)
print('  context_chars: ' + str(len(ctx['text'])))
print('  retrieved_titles: ' + str(ctx['retrieved_titles']))

In [ ]:
PROMPT_TEMPLATE = '''Answer the user's question.

If context is provided, use it and cite specific paper titles and years when available.
If no context is provided, answer from your own knowledge.
Do not invent citations or attributes.

## Context
{context}

## Question
{question}

## Answer'''

def chat(question: str) -> dict:
    ctx_result = get_context(question)
    prompt = PROMPT_TEMPLATE.format(context=ctx_result['text'], question=question)
    messages = [{'role': 'user', 'content': prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors='pt').to(model.device)
    out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=DO_SAMPLE)
    answer = tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    return {
        'answer': answer,
        'retrieved_titles': ctx_result['retrieved_titles'],
        'context_chars': len(ctx_result['text']),
    }

In [ ]:
# preflight
for _var in ['tokenizer', 'model', 'TEST_CASES', 'chat', 'evaluate']:
    if _var not in dir():
        raise NameError('preflight failed: ' + _var + ' not defined — run required cells first.')

results = []
for case in TEST_CASES:
    q_id = case['id']
    q_text = case['question'][:55]
    print('[' + str(q_id).rjust(2) + '/10] ' + q_text + '...', end=' ')
    cfg = {'model': MODEL_NAME, 'max_new_tokens': MAX_NEW_TOKENS, 'do_sample': DO_SAMPLE}
    t0 = time.time()
    chat_result = chat(case['question'])
    latency = time.time() - t0
    results.append({
        'id': case['id'],
        'question': case['question'],
        'answer': chat_result['answer'],
        'latency': latency,
        'config': cfg,
        'case': case,
        'retrieved_titles': chat_result['retrieved_titles'],
        'context_chars': chat_result['context_chars'],
    })
    t_preview = chat_result['retrieved_titles'][:2]
    print('OK ' + str(round(latency, 1)) + 's  ctx=' + str(chat_result['context_chars']) + ' chars  titles=' + str(t_preview))

total = sum(r['latency'] for r in results)
print('Done.  Total: ' + str(round(total, 1)) + 's  |  Avg: ' + str(round(total/len(results), 1)) + 's')

In [ ]:
import pandas as pd

rows = []
for r in results:
    m = evaluate(r)
    rows.append({
        'id': r['id'],
        'question': r['question'][:45],
        'latency_s': round(m['latency_seconds'], 2),
        'chars': m['chars'],
        'kw_recall': round(m['keyword_recall'], 2),
        'kw_hit': str(m['keyword_hit']) + '/' + str(m['keyword_total']),
        'title_hit': m['title_hit'],
        'cite_marker': m['citation_marker_present'],
        'year': m['has_year'],
        'unsure': m['unsure_count'],
        'retrieval_hit': m['retrieval_hit'],
    })

df = pd.DataFrame(rows)
print(df.to_string(index=False))

ms = [evaluate(r) for r in results]
n = len(ms)
print()
print('mean_keyword_recall        : ' + str(round(sum(m['keyword_recall'] for m in ms)/n, 3)))
print('title_hit_rate             : ' + str(round(sum(m['title_hit'] for m in ms)/n, 3)))
print('citation_marker_present_rate: ' + str(round(sum(m['citation_marker_present'] for m in ms)/n, 3)))
print('mean_latency_seconds       : ' + str(round(sum(m['latency_seconds'] for m in ms)/n, 2)))
print('retrieval_hit_rate         : ' + str(round(sum(m['retrieval_hit'] for m in ms)/n, 3)))

for r in results:
    m = evaluate(r)
    case = r['case']
    print()
    print('=' * 68)
    print('[' + str(r['id']) + '] ' + r['question'])
    print('  latency=' + str(round(m['latency_seconds'], 1)) + 's  chars=' + str(m['chars']) + '  words=' + str(m['words']))
    print('  kw_recall=' + str(round(m['keyword_recall'], 2)) + ' (' + str(m['keyword_hit']) + '/' + str(m['keyword_total']) + ')')
    print('  title_hit=' + str(m['title_hit']) + '  cite_marker=' + str(m['citation_marker_present']) + ' (year=' + str(m['has_year']) + ', author=' + str(m['has_author']) + ')')
    print('  unsure=' + str(m['unsure_count']))
    print('  retrieval_hit=' + str(round(m['retrieval_hit'], 1)) + '  (expected: ' + case['expected_paper_title'][:50] + ')')
    print('  retrieved_titles: ' + str(r['retrieved_titles']))
    print('  context_chars: ' + str(r['context_chars']))
    print('  A: ' + r['answer'])